In [19]:
import lmstudio as lms
import pandas as pd
import os
from PIL import Image
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder
import time
from colorama import Fore, Style
import re
from datetime import date

In [20]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # Only use the 2nd GPU

In [22]:
# Qwen
#MODEL_NAME = "qwen/qwen2.5-vl-7b"
#MODEL_NAME = "qwen/qwen3-vl-4b"
#MODEL_NAME = "qwen/qwen3-vl-8b"
#MODEL_NAME = "qwen/qwen3-vl-30b"

# Gemma
#MODEL_NAME = "google/gemma-3-4b"
#MODEL_NAME = "google/gemma-3-12b"
MODEL_NAME = "google/gemma-3-27b"

# LMF2 LiquidAI
#MODEL_NAME = "liquidai_lfm2-vl-450m"
#MODEL_NAME = "lfm2-vl-1.6b"
#MODEL_NAME = "lfm2-vl-3b"

# Llava
#MODEL_NAME = "Llava 8b" # Llava stands for Large Language and Vision Assistant, and is not a standalone Vision Language model

# MistralAI
#MODEL_NAME = "mistralai/magistral-small-2509" # This is a reasoning model, which is expected to be slower

#model = lms.llm("llava") # Llava is not working for some reason
model = lms.llm(MODEL_NAME)

In [23]:
# Example

sample_img = "1JQk5NF.png" # Actual label is 1

In [ ]:
# Setup file paths
IMG_DIR   = "/home/sw_nsolagratia/Documents/Multimodal Project/MultiOFF_Dataset/Labelled Images/"
test_dataset = "/home/sw_nsolagratia/Documents/Multimodal Project/MultiOFF_Dataset/Split Dataset/Testing_meme_dataset.csv"


In [25]:
def strip_thinking(text: str) -> str:
    # Remove any reasoning or thinking blocks, case-insensitive
    cleaned = re.sub(
        r"[\[<]\s*think\s*[\]>].*?[\[<]\s*/\s*think\s*[\]>]",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    return cleaned.strip()

In [26]:
# Warm up

result = model.respond("Hi how are you")
print(result)

I am doing well, thank you for asking! As an AI, I don't *feel* in the same way people do, but my systems are running smoothly and I'm ready to help. 😄 

How are *you* doing today? Is there anything I can assist you with?


In [27]:
result = str(result)
print(strip_thinking(result))

I am doing well, thank you for asking! As an AI, I don't *feel* in the same way people do, but my systems are running smoothly and I'm ready to help. 😄 

How are *you* doing today? Is there anything I can assist you with?


In [28]:
example_path = os.path.join(IMG_DIR, sample_img)
example_img_handle = lms.prepare_image(example_path)

In [29]:
# Warming up with image input
# Actual / Correct label is 1

for i in range(10):
    chat = lms.Chat()

    chat.add_user_message(
        "Reply with ONLY 1 for Yes OR 0 for NO. Nothing else. Do not describe the image. The question being: is this image considered hate speech or offensive?",
        images=[example_img_handle]
    )
    example_response = str(model.respond(chat))
    example_response = strip_thinking(example_response) 
    print(f"Response no. {i+1}\t: {example_response}")

    del chat

Response no. 1	: 0
Response no. 2	: 0
Response no. 3	: 0
Response no. 4	: 0
Response no. 5	: 0
Response no. 6	: 0
Response no. 7	: 0
Response no. 8	: 0
Response no. 9	: 0
Response no. 10	: 0


In [30]:
#model.unload() 

In [31]:
# Setup dataset
#df = pd.read_csv(dataset) # Entire dataset
df = pd.read_csv(test_dataset) # Test dataset
display(df) # before
# Label encoding
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'].tolist())
filenames = df['image_name'].tolist()
actual_labels = df['label'].tolist()

,image_name,sentence,label
0,jyxHhiB.png,3 hrs Black nurse in Connecticut asked me if T...,offensive
1,we4hhWi.png,"I do n't believe that women have any rights , ...",offensive
2,TpIZoZr.png,WHY THE FUCK DOES IRITHE DONALD HAVE BETTER NE...,offensive
3,h6Pkqkr.png,"After the Bern subsides , get ready to ... Fee...",Non-offensiv
4,94anjQG.png,"MAG & WALLY Mrs. Clinton , I 'm awesome ! cong...",Non-offensiv
...,...,...,...
143,fyAh3I0.png,WHAT IF POKEMON GO WAS RELEASED TO D US FRO TH...,Non-offensiv
144,CMQqhImUkAAKeno.jpg,Donald Trump 's hair looks like someone tried ...,Non-offensiv
145,tINjUCc.png,MAMAS Who am I supposed to vote for ? Am l sup...,Non-offensiv
146,eI2N5iQ.png,EcakpnBeHn rnacoBnl 18 minutes ago We will hav...,Non-offensiv


In [32]:
display(df) # after

,image_name,sentence,label
0,jyxHhiB.png,3 hrs Black nurse in Connecticut asked me if T...,1
1,we4hhWi.png,"I do n't believe that women have any rights , ...",1
2,TpIZoZr.png,WHY THE FUCK DOES IRITHE DONALD HAVE BETTER NE...,1
3,h6Pkqkr.png,"After the Bern subsides , get ready to ... Fee...",0
4,94anjQG.png,"MAG & WALLY Mrs. Clinton , I 'm awesome ! cong...",0
...,...,...,...
143,fyAh3I0.png,WHAT IF POKEMON GO WAS RELEASED TO D US FRO TH...,0
144,CMQqhImUkAAKeno.jpg,Donald Trump 's hair looks like someone tried ...,0
145,tINjUCc.png,MAMAS Who am I supposed to vote for ? Am l sup...,0
146,eI2N5iQ.png,EcakpnBeHn rnacoBnl 18 minutes ago We will hav...,0


In [33]:
# Set the batch size to fit the token limit
batch_size = 10

# Split the filenames into batches
batches = [filenames[i:i + batch_size] for i in range(0, len(filenames), batch_size)]

label_map = dict(zip(df['image_name'], df['label'])) # Map filenames to actual labels

safe_model_name = re.sub(r'[\\/*?:"<>|/]', '_', MODEL_NAME)

evaluation_file_name = f"{safe_model_name}_results.csv"

In [ ]:
# Evaluate the model 30 times

for i in range(1, 31):
    print(f"{Style.BRIGHT}{Fore.LIGHTGREEN_EX}Run no: {i}{Style.RESET_ALL}")
    # Start Timer
    start_time = time.perf_counter()

    processed_images = set()
    results_list = []
    predictions = []

    for batch in batches:
        try:
            for filename in batch:
                # Reset chat for each batch, reset again per image, ensuring a clean start
                chat = lms.Chat()
                if filename in processed_images:
                    continue

                image_path = os.path.join(IMG_DIR, filename)
                image_handle = lms.prepare_image(image_path)

                processed_images.add(filename)

                chat.add_user_message(
                    "Reply with ONLY 1 for Yes OR 0 for NO. Nothing else. Do not describe the image. The question being: is this image considered hate speech or offensive?",
                    images=[image_handle]
                )

                # Get the response for the single image immediately
                response = str(model.respond(chat))
                response = strip_thinking(response)

                # Normalize and handle the single response immediately
                single_prediction = -1
                
                if isinstance(response, list):
                    if response and response[0].strip() in ["0", "1"]:
                        single_prediction = int(response[0].strip())
                elif response in ["0", "1"]:
                    single_prediction = int(response)

                # Collect and store the result
                predictions.append(single_prediction) 
                actual_label = label_map.get(filename, 'N/A')

                results_list.append({
                    'filename': filename,
                    'actual_label': actual_label,
                    'predicted_label': single_prediction
                })

            # Reset the chat history for the next batch after processing all images
            chat = lms.Chat()

        except Exception as e:
            print(f"{Style.BRIGHT}{Fore.RED}Error occurred during batch processing: {e}{Style.RESET_ALL}")
            model.unload()
            break

    # End Timer
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    # Create the final DataFrame from the list of results
    results_df = pd.DataFrame(results_list)

    # Print Elapsed Time 
    hours, remainder = divmod(elapsed_time, 3600)
    minutes, seconds = divmod(remainder, 60)
    print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Evaluation Time: {int(hours):02d}h {int(minutes):02d}m {seconds:05.2f}s{Style.RESET_ALL}")

    # Final accuracy calculation
    # Filter out samples where prediction failed (-1) before calculating accuracy
    valid_results_df = results_df[results_df['predicted_label'].isin([0, 1])].copy()
    valid_predictions = valid_results_df['predicted_label'].tolist()
    valid_actuals = valid_results_df['actual_label'].tolist()

    print(f"{Style.BRIGHT}{Fore.BLUE}Total processed images: {len(results_df)}{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.GREEN}Valid predictions used for metrics: {len(valid_predictions)}{Style.RESET_ALL}")
    time_str = f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}" 
    print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Evaluation Time: {time_str}{Style.RESET_ALL}")


    if len(valid_predictions) > 0:
        # Prepare the list for corrected predictions, treating -1 as incorrect
        results_df['predicted_label_corrected'] = results_df['predicted_label'].apply(lambda x: x if x != -1 else None)

        # Filter out rows where predicted_label is None (i.e., -1 values)
        valid_results_df = results_df[results_df['predicted_label_corrected'].notna()]

        valid_predictions = valid_results_df['predicted_label_corrected'].tolist()
        valid_actuals = valid_results_df['actual_label'].tolist()

        # Calculate metrics, treating -1 as incorrect but keeping 0 as valid if it matches the actual label
        accuracy = accuracy_score(valid_actuals, valid_predictions)
        mcc = matthews_corrcoef(valid_actuals, valid_predictions)

        # Calculate correct predictions
        correct_predictions = sum(1 for p, a in zip(valid_predictions, valid_actuals) if p == a)

        today = date.today()
        formatted_date = today.strftime("%Y-%m-%d")

        # Prepare the evaluation results dictionary
        evaluation_results = {
            "Model_Name": MODEL_NAME, 
            "Accuracy": accuracy,
            "MCC": mcc,
            "Time": time_str,
            "Date": formatted_date
        }

        eval_df = pd.DataFrame([evaluation_results])

        # Check if the CSV file already exists
        if os.path.exists(evaluation_file_name):
            # If the file exists, append the new results to it
            eval_df.to_csv(evaluation_file_name, mode='a', header=False, index=False)
            print(f"\nAppended evaluation results to {evaluation_file_name}")
        else:
            # If the file does not exist, create a new one and add header
            eval_df.to_csv(evaluation_file_name, mode='w', header=True, index=False)
            print(f"\nCreated new evaluation file: {evaluation_file_name}")

        del evaluation_results, accuracy, mcc, valid_actuals, valid_predictions, valid_results_df, results_df, today, formatted_date, processed_images, results_list, predictions
    else:
        print(f"{Style.BRIGHT}{Fore.RED}No valid predictions (0 or 1) were collected for metrics calculation.{Style.RESET_ALL}")

Run no: 1
Total Evaluation Time: 00h 01m 58.91s
Total processed images: 148
Valid predictions used for metrics: 148
Total Evaluation Time: 00:01:58

Created new evaluation file: google_gemma-3-27b_results.csv
Run no: 2
Total Evaluation Time: 00h 02m 10.09s
Total processed images: 148
Valid predictions used for metrics: 148
Total Evaluation Time: 00:02:10

Appended evaluation results to google_gemma-3-27b_results.csv
Run no: 3
Total Evaluation Time: 00h 02m 05.24s
Total processed images: 148
Valid predictions used for metrics: 148
Total Evaluation Time: 00:02:05

Appended evaluation results to google_gemma-3-27b_results.csv
Run no: 4
Total Evaluation Time: 00h 02m 06.19s
Total processed images: 148
Valid predictions used for metrics: 148
Total Evaluation Time: 00:02:06

Appended evaluation results to google_gemma-3-27b_results.csv
Run no: 5
Total Evaluation Time: 00h 02m 08.37s
Total processed images: 148
Valid predictions used for metrics: 148
Total Evaluation Time: 00:02:08

Appended 

In [35]:
model.unload() # Eject the model from memory
print("Model unloaded")

Model unloaded


In [36]:
variances_df = pd.read_csv(evaluation_file_name)

summary_df = variances_df.describe()
display(summary_df)
summary_df.to_csv(f'{safe_model_name}_summary.csv', mode='w', header=True, index=False)
print(f"\nCreated new evaluation file: {MODEL_NAME}_summary.csv")

,Accuracy,MCC
count,30.000000,30.000000
mean,0.686036,0.309871
std,0.003860,0.008714
min,0.675676,0.285729
25%,0.682432,0.302572
50%,0.689189,0.316754
75%,0.689189,0.316754
max,0.689189,0.319258



Created new evaluation file: google/gemma-3-27b_summary.csv
